# BLEVE Pressure Intelligence

**Physics-guided ensemble learning for peak blast-pressure estimation**

This notebook presents a reproducible regression workflow for estimating maximum pressure in simulated Boiling Liquid Expanding Vapour Explosion (BLEVE) scenarios. It combines transparent data-quality checks, domain-guided feature construction, model-family comparison, cross-validated tuning, nonlinear ensembling, conservative residual calibration, and prediction-contract validation.

## Recorded benchmark evidence

- Out-of-fold MAPE: **0.08919**
- Out-of-fold R²: **0.97477**
- Public holdout MAPE: **0.16233**
- Selected model: HistGradientBoosting and MLP ensemble with cross-validated Huber residual calibration

The accompanying repository separates reusable feature and validation logic into a tested Python package. Raw benchmark data is intentionally excluded because redistribution terms were not supplied with the dataset.

> **Safety boundary:** This is a research and simulation workflow. It is not a live monitoring product, a certified engineering model, or a substitute for validated process-safety analysis.

## Workflow

1. Load and audit scenario data.
2. Remove invalid targets and exact duplicate records.
3. Construct geometry, thermodynamic, obstacle, and sensor features.
4. Compare distinct regression families under cross-validation.
5. Tune HistGradientBoosting and MLP candidates.
6. Blend complementary nonlinear predictions.
7. Apply bounded, cross-fitted Huber residual calibration.
8. Validate the final export contract and record evidence artifacts.


In [ ]:
from pathlib import Path
import multiprocessing
import os
import re
import shutil
import sys
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from joblib import Parallel, delayed
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.svm import LinearSVR

try:
    from threadpoolctl import threadpool_limits
except ImportError:
    threadpool_limits = None


def discover_project_root() -> Path:
    configured = os.environ.get("BLEVE_PROJECT_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        if not (root / "pyproject.toml").exists():
            raise FileNotFoundError(f"BLEVE_PROJECT_ROOT is not a project root: {root}")
        return root

    current = Path.cwd().resolve()
    for candidate in (current, current.parent, current.parent.parent):
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the repository root. Start Jupyter from this repository "
        "or set BLEVE_PROJECT_ROOT."
    )


PROJECT_ROOT = discover_project_root()
SOURCE_DIR = PROJECT_ROOT / "src"
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

from bleve_pressure import add_physics_features, validate_prediction_frame

RANDOM_STATE = 42
ID_COL = "ID"
TARGET_COL = "Target Pressure (bar)"
N_SPLITS_COMPARISON = 3
N_SPLITS_FINAL = 5

CPU_COUNT = os.cpu_count() or multiprocessing.cpu_count()
RUN_LOCAL_HIGH_CPU = "COLAB_RELEASE_TAG" not in os.environ
PARALLEL_JOBS = (
    max(2, min(12, int(max(2, CPU_COUNT) * 0.75)))
    if RUN_LOCAL_HIGH_CPU
    else max(1, min(4, CPU_COUNT // 2))
)
PARALLEL_BACKEND_PREFER = "threads"
CPU_INNER_THREADS = 1
USE_PARALLEL_CANDIDATE_EVAL = True

np.random.seed(RANDOM_STATE)
print("Notebook configured.")
print("Detected CPU threads:", CPU_COUNT)
print("Parallel jobs selected:", PARALLEL_JOBS)


## 1. Load training and test data

The notebook expects `train.csv` and `test.csv` to be in the same folder as `main.ipynb`.

The training file contains the target pressure values used for model fitting and validation. The test file contains the feature values for the rows where final predictions are required.

In [ ]:
def resolve_path_from_project(value: str | None, default: Path) -> Path:
    if value is None:
        return default.resolve()
    configured = Path(value).expanduser()
    return (configured if configured.is_absolute() else PROJECT_ROOT / configured).resolve()


def reset_output_folder(path: Path) -> None:
    if path.exists():
        try:
            shutil.rmtree(path)
        except PermissionError as exc:
            raise PermissionError(
                f"Could not reset {path}. Close files opened from this folder and rerun."
            ) from exc
    (path / "tables").mkdir(parents=True, exist_ok=True)
    (path / "figures").mkdir(parents=True, exist_ok=True)


DATA_DIR = resolve_path_from_project(os.environ.get("BLEVE_DATA_DIR"), PROJECT_ROOT / "data")
OUTPUT_DIR = resolve_path_from_project(
    os.environ.get("BLEVE_ARTIFACT_DIR"),
    PROJECT_ROOT / "artifacts" / "runs" / "latest",
)
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
SUMMARY_DIR = TABLE_DIR
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"

missing_files = [str(path) for path in (TRAIN_PATH, TEST_PATH) if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "Required local benchmark files are unavailable: "
        + ", ".join(missing_files)
        + ". See data/README.md for the expected schema and data boundary."
    )

reset_output_folder(OUTPUT_DIR)
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)

print("Scenario data loaded.")
print("Training shape:", train_raw.shape)
print("Test shape:", test_raw.shape)
print("Artifacts:", OUTPUT_DIR.relative_to(PROJECT_ROOT))

display(train_raw.head())
display(test_raw.head())


## 2. Data audit

Initial data quality checks are performed before modelling because missing values, duplicate rows, inconsistent category labels, unusual target values, and possible outliers can affect model performance.

The column names and value ranges are also reviewed before feature engineering, because the dataset documentation and the dataset column names may use slightly different wording.

In [ ]:
audit_rows = [
    {"item": "train_rows", "value": len(train_raw)},
    {"item": "test_rows", "value": len(test_raw)},
    {"item": "train_columns", "value": train_raw.shape[1]},
    {"item": "test_columns", "value": test_raw.shape[1]},
    {"item": "target_present_in_train", "value": TARGET_COL in train_raw.columns},
    {"item": "target_present_in_test", "value": TARGET_COL in test_raw.columns},
    {"item": "duplicate_train_rows", "value": int(train_raw.duplicated().sum())},
    {"item": "duplicate_test_rows", "value": int(test_raw.duplicated().sum())},
]

if TARGET_COL in train_raw.columns:
    target_numeric = pd.to_numeric(train_raw[TARGET_COL], errors="coerce")
    audit_rows.append({"item": "missing_target_rows", "value": int(target_numeric.isna().sum())})
    audit_rows.append({"item": "non_positive_target_rows", "value": int((target_numeric <= 0).sum())})
    audit_rows.append({"item": "target_min", "value": float(target_numeric.min())})
    audit_rows.append({"item": "target_max", "value": float(target_numeric.max())})
    audit_rows.append({"item": "target_median", "value": float(target_numeric.median())})

if "Tank Failure Pressure (bar)" in train_raw.columns:
    pressure_numeric = pd.to_numeric(train_raw["Tank Failure Pressure (bar)"], errors="coerce")
    audit_rows.append({"item": "tank_failure_pressure_min", "value": float(pressure_numeric.min())})
    audit_rows.append({"item": "tank_failure_pressure_max", "value": float(pressure_numeric.max())})
    audit_rows.append({"item": "tank_failure_pressure_missing_rows", "value": int(pressure_numeric.isna().sum())})

if "Status" in train_raw.columns:
    status_values = sorted(train_raw["Status"].dropna().astype(str).unique())
    audit_rows.append({"item": "status_unique_values", "value": ", ".join(status_values)})

audit_summary = pd.DataFrame(audit_rows)
display(audit_summary)
audit_summary.to_csv(TABLE_DIR / "00_data_audit.csv", index=False)

missing_summary = (
    train_raw.isna().sum().rename("train_missing")
    .to_frame()
    .join(test_raw.isna().sum().rename("test_missing"), how="outer")
    .fillna(0)
    .astype(int)
    .sort_values(["train_missing", "test_missing"], ascending=False)
)

display(missing_summary.head(25))
missing_summary.to_csv(TABLE_DIR / "00_missing_value_summary.csv")

if TARGET_COL not in train_raw.columns:
    raise KeyError(f"Missing target column: {TARGET_COL}")

# Rows with invalid target values are excluded before supervised model fitting.
# Feature missing values are handled inside the preprocessing pipelines.
train_clean = train_raw.copy()
train_clean[TARGET_COL] = pd.to_numeric(train_clean[TARGET_COL], errors="coerce")

valid_target_mask = (
    train_clean[TARGET_COL].notna()
    & np.isfinite(train_clean[TARGET_COL])
    & (train_clean[TARGET_COL] > 0)
)

invalid_target_count = int((~valid_target_mask).sum())
train_clean = train_clean.loc[valid_target_mask].reset_index(drop=True)

# Duplicate rows are being removed after invalid target rows are excluded from it.so this keeps the supervised training data clean and avoids repeated records biasing the model
duplicate_rows_after_target_cleaning = int(train_clean.duplicated().sum())
before_duplicate_removal = len(train_clean)
train_clean = train_clean.drop_duplicates().reset_index(drop=True)
duplicate_rows_removed = before_duplicate_removal - len(train_clean)

test_clean = test_raw.copy().reset_index(drop=True)

cleaning_action_summary = pd.DataFrame([
    {
        "cleaning_step": "Invalid target removal",
        "rows_affected": invalid_target_count,
        "action_taken": "Rows with missing, infinite, or non-positive target pressure were removed before modelling.",
    },
    {
        "cleaning_step": "Duplicate row removal",
        "rows_affected": duplicate_rows_removed,
        "action_taken": "Exact duplicate training rows were removed after target cleaning to avoid repeated-record bias.",
    },
])

display(cleaning_action_summary)
cleaning_action_summary.to_csv(TABLE_DIR / "00_cleaning_action_summary.csv", index=False)

y = train_clean[TARGET_COL].astype(float).values

print("Invalid target rows removed:", invalid_target_count)
print("Training rows used for modelling:", len(train_clean))
print("Target summary after cleaning:")
display(pd.Series(y, name=TARGET_COL).describe())

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(y, bins=60)
plt.xlabel("Target Pressure (bar)")
plt.ylabel("Count")
plt.title("Training Target Pressure Distribution")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "target_distribution.png", dpi=150)
plt.show()

if "Status" in train_raw.columns:
    status_counts = (
        train_raw["Status"]
        .value_counts(dropna=False)
        .rename_axis("status_value")
        .reset_index(name="count")
    )
    display(status_counts)
    status_counts.to_csv(SUMMARY_DIR / "status_value_counts.csv", index=False)

if "Tank Failure Pressure (bar)" in train_raw.columns:
    pressure_values = pd.to_numeric(train_raw["Tank Failure Pressure (bar)"], errors="coerce")
    pressure_check = pd.DataFrame({
        "statistic": ["min", "median", "mean", "p95", "p99", "max", "missing_rows"],
        "value": [
            float(pressure_values.min()),
            float(pressure_values.median()),
            float(pressure_values.mean()),
            float(pressure_values.quantile(0.95)),
            float(pressure_values.quantile(0.99)),
            float(pressure_values.max()),
            int(pressure_values.isna().sum()),
        ],
    })
    display(pressure_check)
    pressure_check.to_csv(SUMMARY_DIR / "tank_failure_pressure_summary.csv", index=False)

## 3. Preprocessing and feature engineering

The engineered features are based on the physical setup of the BLEVE scenario, including tank geometry, pressure, temperature, obstacle geometry, obstacle angle, and sensor position.

The feature engineering step keeps the derived variables interpretable so that the modelling process can be explained clearly. The scale and unit meaning of engineered features are also reviewed before modelling. For example, ratio variables are handled as numeric proportions, and temperature-related features are created using the value ranges observed in the dataset.

Small safeguards are included in the feature engineering code so that the notebook can handle minor differences in column naming without stopping unexpectedly.

In [ ]:
X_train_fe = add_physics_features(
    train_clean.drop(columns=[TARGET_COL], errors="ignore")
)
X_test_fe = add_physics_features(test_clean.copy())

for column in [name for name in X_train_fe.columns if name not in X_test_fe.columns]:
    X_test_fe[column] = np.nan
for column in [name for name in X_test_fe.columns if name not in X_train_fe.columns]:
    X_train_fe[column] = np.nan

X_test_fe = X_test_fe[X_train_fe.columns]
test_ids = (
    X_test_fe[ID_COL].values
    if ID_COL in X_test_fe.columns
    else test_clean[ID_COL].values
)
X_train_fe = X_train_fe.drop(columns=[ID_COL], errors="ignore")
X_test_fe = X_test_fe.drop(columns=[ID_COL], errors="ignore")

print("Feature-engineered training shape:", X_train_fe.shape)
print("Feature-engineered test shape:", X_test_fe.shape)
display(X_train_fe.head())


In [ ]:
# Feature relationship plots
relationship_features = [
    "FE_sensor_3d_distance_bleve",
    "FE_pressure_over_distance",
    "FE_inv_sensor_distance_sq",
    "FE_obstacle_area_over_distance",
    "FE_tank_volume_box",
]

for feature in relationship_features:
    if feature in X_train_fe.columns:
        feature_values = pd.to_numeric(X_train_fe[feature], errors="coerce")

        plt.figure(figsize=(6, 4))
        plt.scatter(feature_values, y, alpha=0.25, s=10)
        plt.xlabel(feature)
        plt.ylabel(TARGET_COL)
        plt.title(f"{feature} and Target Pressure")
        plt.tight_layout()

        safe_name = re.sub(r"[^A-Za-z0-9]+", "_", feature).strip("_").lower()
        plt.savefig(FIGURE_DIR / f"scatter_{safe_name}.png", dpi=150)
        plt.show()

distribution_features = [
    "Tank Failure Pressure (bar)",
    "FE_liquid_ratio_fraction",
    "FE_sensor_3d_distance_bleve",
    "FE_pressure_over_distance",
]

for feature in distribution_features:
    if feature in X_train_fe.columns and feature in X_test_fe.columns:
        train_values = pd.to_numeric(X_train_fe[feature], errors="coerce").dropna()
        test_values = pd.to_numeric(X_test_fe[feature], errors="coerce").dropna()

        plt.figure(figsize=(7, 4))
        plt.hist(train_values, bins=50, alpha=0.55, label="training")
        plt.hist(test_values, bins=50, alpha=0.55, label="test")
        plt.xlabel(feature)
        plt.ylabel("Count")
        plt.title(f"Training and Test Feature Distribution: {feature}")
        plt.legend()
        plt.tight_layout()

        safe_name = re.sub(r"[^A-Za-z0-9]+", "_", feature).strip("_").lower()
        plt.savefig(FIGURE_DIR / f"train_test_{safe_name}.png", dpi=150)
        plt.show()

### Feature selection review
This table reviews numeric engineered features using their absolute correlation with the training target. The table is used as supporting evidence and does not remove features from the final modelling pipeline.

In [ ]:
numeric_feature_review = []

for column in X_train_fe.columns:
    if pd.api.types.is_numeric_dtype(X_train_fe[column]):
        values = pd.to_numeric(X_train_fe[column], errors="coerce")
        if values.notna().sum() > 5 and values.nunique(dropna=True) > 1:
            corr = np.corrcoef(values.fillna(values.median()), y)[0, 1]
            numeric_feature_review.append({
                "feature": column,
                "abs_correlation_with_target": float(abs(corr)) if np.isfinite(corr) else np.nan,
                "missing_values": int(values.isna().sum()),
                "unique_values": int(values.nunique(dropna=True)),
            })

feature_selection_review = (
    pd.DataFrame(numeric_feature_review)
    .sort_values("abs_correlation_with_target", ascending=False)
    .reset_index(drop=True)
)

display(feature_selection_review.head(30))
feature_selection_review.to_csv(TABLE_DIR / "01_feature_selection_review.csv", index=False)

## 4. Preprocessing utilities

The models use slightly different preprocessing steps based on their modelling assumptions:

- linear, SVR, and MLP models use imputation, categorical one-hot encoding, and feature scaling
- the HGB pipeline uses imputation and ordinal encoding, because tree-based models do not require standard scaling

In [ ]:
EXPLICIT_CATEGORICAL = {"Status", "Sensor ID", "Sensor Position Side", "Thermodynamic Profile"}

def identify_columns(X):
    categorical = []
    for column in X.columns:
        if (
            X[column].dtype == "object"
            or column in EXPLICIT_CATEGORICAL
            or column.startswith("FE_KEY_")
        ):
            categorical.append(column)

    numeric = [c for c in X.columns if c not in categorical]
    return numeric, categorical

numeric_cols, categorical_cols = identify_columns(X_train_fe)

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))
print("Categorical column names:")
print(categorical_cols)

for column in numeric_cols:
    X_train_fe[column] = pd.to_numeric(X_train_fe[column], errors="coerce")
    X_test_fe[column] = pd.to_numeric(X_test_fe[column], errors="coerce")

for column in categorical_cols:
    X_train_fe[column] = X_train_fe[column].astype(str).fillna("missing")
    X_test_fe[column] = X_test_fe[column].astype(str).fillna("missing")

def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)

def make_scaled_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]), numeric_cols),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", make_ohe()),
            ]), categorical_cols),
        ],
        remainder="drop",
    )

def make_hgb_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("num", SimpleImputer(strategy="median"), numeric_cols),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
            ]), categorical_cols),
        ],
        remainder="drop",
    )

def make_dynamic_scaled_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]), make_column_selector(dtype_include=np.number)),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", make_ohe()),
            ]), make_column_selector(dtype_exclude=np.number)),
        ],
        remainder="drop",
    )

def make_dynamic_hgb_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("num", SimpleImputer(strategy="median"), make_column_selector(dtype_include=np.number)),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
            ]), make_column_selector(dtype_exclude=np.number)),
        ],
        remainder="drop",
    )

class SmoothedTargetEncoderFeatures(BaseEstimator, TransformerMixin):
    # This transformer creates smoothed category-level target summaries during model fitting.
    def __init__(self, columns=None, smoothing=30.0, prefix="TE_"):
        self.columns = columns
        self.smoothing = smoothing
        self.prefix = prefix

    def fit(self, X, y):
        X_df = pd.DataFrame(X).copy()
        y_series = pd.Series(y).reset_index(drop=True)
        self.global_mean_ = float(y_series.mean())
        self.columns_ = [c for c in (self.columns or []) if c in X_df.columns]
        self.mappings_ = {}

        for col in self.columns_:
            keys = X_df[col].astype(str).fillna("missing").reset_index(drop=True)
            tmp = pd.DataFrame({"key": keys, "target": y_series})
            stats = tmp.groupby("key")["target"].agg(["mean", "count"])
            smooth = (
                stats["mean"] * stats["count"]
                + self.global_mean_ * self.smoothing
            ) / (stats["count"] + self.smoothing)
            self.mappings_[col] = smooth.to_dict()

        return self

    def transform(self, X):
        X_df = pd.DataFrame(X).copy()

        for col in getattr(self, "columns_", []):
            safe_name = re.sub(r"[^A-Za-z0-9]+", "_", str(col)).strip("_")
            encoded = X_df[col].astype(str).fillna("missing").map(self.mappings_.get(col, {}))
            X_df[f"{self.prefix}{safe_name}"] = encoded.fillna(self.global_mean_).astype(float)

        return X_df

TARGET_ENCODING_COLUMNS = [
    c for c in [
        "Status",
        "Sensor ID",
        "Sensor Position Side",
        "Thermodynamic Profile",
        "FE_KEY_sensor_side",
        "FE_KEY_sensor_status",
        "FE_KEY_side_status",
        "FE_KEY_thermo_status",
        "FE_KEY_sensor_side_status",
    ]
    if c in X_train_fe.columns
]

print("Target encoding columns:", TARGET_ENCODING_COLUMNS)

log_y = np.log(y)
log1p_y = np.log1p(y)

# These sample weights support MAPE-focused training by giving lower target values more influence.
sqrt_weights = 1.0 / np.sqrt(np.clip(y, 0.03, None))
sqrt_weights = np.clip(sqrt_weights, 0.35, 5.0)
sqrt_weights = sqrt_weights / np.mean(sqrt_weights)

mape_weights = 1.0 / np.clip(y, 0.03, None)
mape_weights = np.clip(mape_weights, 0.25, 8.0)
mape_weights = mape_weights / np.mean(mape_weights)

# This weighting option balances low-pressure accuracy with high-pressure stability.
soft_low_pressure_weights = 1.0 / np.power(np.clip(y, 0.035, None), 0.35)
soft_low_pressure_weights = np.clip(soft_low_pressure_weights, 0.40, 3.75)
soft_low_pressure_weights = soft_low_pressure_weights / np.mean(soft_low_pressure_weights)

## 5. Validation strategy and metrics

MAPE is used as the main error metric because it is the required prediction-performance metric for this task.

R2, RMSE, and MAE are also reported to give a more complete comparison of model performance.

In [ ]:
def get_splits(n_splits, seed=RANDOM_STATE):
    try:
        bins = pd.qcut(y, q=20, labels=False, duplicates="drop")
        splitter = StratifiedKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=seed,
        )
        splits = list(splitter.split(np.zeros(len(y)), bins))
        return splits, "Target stratified KFold"
    except Exception:
        splitter = KFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=seed,
        )
        splits = list(splitter.split(X_train_fe, y))
        return splits, "KFold"

comparison_splits, comparison_cv_name = get_splits(N_SPLITS_COMPARISON)
final_splits, final_cv_name = get_splits(N_SPLITS_FINAL)

print("Comparison CV:", comparison_cv_name, "folds:", len(comparison_splits))
print("Final CV:", final_cv_name, "folds:", len(final_splits))

def regression_scores(y_true, pred):
    pred = np.clip(np.asarray(pred, dtype=float), 1e-9, None)
    return {
        "mape": float(mean_absolute_percentage_error(y_true, pred)),
        "r2": float(r2_score(y_true, pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, pred))),
        "mae": float(mean_absolute_error(y_true, pred)),
    }

def fold_mape_std(pred, splits):
    values = []
    for _, valid_idx in splits:
        fold_pred = np.clip(pred[valid_idx], 1e-9, None)
        values.append(mean_absolute_percentage_error(y[valid_idx], fold_pred))

    return float(np.mean(values)), float(np.std(values))

def inverse_transform(pred_transformed, target_mode):
    if target_mode == "log":
        return np.exp(pred_transformed)
    if target_mode == "log1p":
        return np.clip(np.expm1(pred_transformed), 1e-9, None)
    return np.asarray(pred_transformed, dtype=float)

def get_target_values(target_mode):
    if target_mode == "log":
        return log_y
    if target_mode == "log1p":
        return log1p_y
    return y

## 6. Model family comparison

Several different model families are evaluated before selecting the final model.

This comparison tests fundamentally different model families under the same validation protocol. It also explains why the HistGradientBoosting model is used as the main model component in the final ensemble.

In [ ]:
def evaluate_pipeline_cv(model_id, family, pipeline, target_mode="log", splits=None, sample_weight=None):
    if splits is None:
        splits = comparison_splits

    target_values = get_target_values(target_mode)
    oof = np.zeros(len(y), dtype=float)

    for train_idx, valid_idx in splits:
        fitted = clone(pipeline)
        fit_kwargs = {}

        if sample_weight is not None:
            fit_kwargs["model__sample_weight"] = sample_weight[train_idx]

        try:
            fitted.fit(X_train_fe.iloc[train_idx], target_values[train_idx], **fit_kwargs)
        except TypeError:
            fitted.fit(X_train_fe.iloc[train_idx], target_values[train_idx])

        valid_pred = fitted.predict(X_train_fe.iloc[valid_idx])
        oof[valid_idx] = inverse_transform(valid_pred, target_mode)

    scores = regression_scores(y, oof)
    fold_mean, fold_std = fold_mape_std(oof, splits)

    return {
        "model_id": model_id,
        "family": family,
        "target_mode": target_mode,
        "fold_mape_mean": fold_mean,
        "fold_mape_std": fold_std,
        **scores,
    }

comparison_rows = []

baseline = Pipeline([
    ("model", DummyRegressor(strategy="median")),
])
comparison_rows.append(
    evaluate_pipeline_cv("DUMMY_MEDIAN", "Baseline", baseline, target_mode="raw")
)

scaled_pre = make_scaled_preprocessor()

comparison_models = [
    (
        "RIDGE_LOG",
        "Regularised Linear",
        Pipeline([
            ("preprocessor", scaled_pre),
            ("model", Ridge(alpha=10.0)),
        ]),
    ),
    (
        "ELASTICNET_LOG",
        "Regularised Linear",
        Pipeline([
            ("preprocessor", scaled_pre),
            ("model", ElasticNet(alpha=0.002, l1_ratio=0.15, max_iter=6000, random_state=RANDOM_STATE)),
        ]),
    ),
    (
        "LINEAR_SVR_LOG",
        "Support Vector Regression",
        Pipeline([
            ("preprocessor", scaled_pre),
            ("model", LinearSVR(C=1.0, epsilon=0.02, max_iter=5000, random_state=RANDOM_STATE)),
        ]),
    ),
    (
        "MLP_LOG",
        "Neural Network",
        Pipeline([
            ("preprocessor", scaled_pre),
            ("model", MLPRegressor(
                hidden_layer_sizes=(80, 40),
                alpha=0.002,
                learning_rate_init=0.002,
                max_iter=350,
                early_stopping=True,
                random_state=RANDOM_STATE,
            )),
        ]),
    ),
]

for model_id, family, pipeline in comparison_models:
    print("Training comparison model:", model_id)
    comparison_rows.append(
        evaluate_pipeline_cv(model_id, family, pipeline, target_mode="log")
    )

hgb_comparison = Pipeline([
    ("preprocessor", make_hgb_preprocessor()),
    ("model", HistGradientBoostingRegressor(
        loss="squared_error",
        max_iter=800,
        learning_rate=0.035,
        max_leaf_nodes=31,
        min_samples_leaf=18,
        l2_regularization=0.03,
        early_stopping=True,
        validation_fraction=0.12,
        n_iter_no_change=40,
        random_state=RANDOM_STATE,
    )),
])

comparison_rows.append(
    evaluate_pipeline_cv(
        "HGB_LOG_WEIGHTED",
        "Gradient Boosted Trees",
        hgb_comparison,
        target_mode="log",
        sample_weight=sqrt_weights,
    )
)

model_family_comparison = (
    pd.DataFrame(comparison_rows)
    .sort_values(["mape", "fold_mape_std"])
    .reset_index(drop=True)
)

display(model_family_comparison)
model_family_comparison.to_csv(TABLE_DIR / "02_model_family_comparison.csv", index=False)

plt.figure(figsize=(9, 4))
plt.bar(model_family_comparison["model_id"], model_family_comparison["mape"])
plt.xticks(rotation=75, ha="right")
plt.ylabel("Validation MAPE")
plt.title("Model Family Comparison by MAPE")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "model_family_mape.png", dpi=150)
plt.show()

plt.figure(figsize=(9, 4))
plt.bar(model_family_comparison["model_id"], model_family_comparison["r2"])
plt.xticks(rotation=75, ha="right")
plt.ylabel("Validation R2")
plt.title("Model Family Comparison by R2")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "model_family_r2.png", dpi=150)
plt.show()

## 7. HistGradientBoosting tuning

HistGradientBoosting is used as the main tabular model component in this notebook.

A small set of nearby hyperparameter settings is tested to keep the tuning controlled, reproducible, and easy to compare.

In [ ]:
def build_hgb_pipeline(params, use_target_encoding=False, te_smoothing=30.0):
    model_params = dict(params)
    model_random_state = model_params.pop("random_state", RANDOM_STATE)

    model = HistGradientBoostingRegressor(
        early_stopping=True,
        validation_fraction=0.12,
        n_iter_no_change=50,
        random_state=model_random_state,
        **model_params,
    )

    steps = []

    if use_target_encoding:
        steps.append((
            "target_encoding",
            SmoothedTargetEncoderFeatures(
                columns=TARGET_ENCODING_COLUMNS,
                smoothing=te_smoothing,
                prefix="TE_",
            ),
        ))
        steps.append(("preprocessor", make_dynamic_hgb_preprocessor()))
    else:
        steps.append(("preprocessor", make_hgb_preprocessor()))

    steps.append(("model", model))
    return Pipeline(steps)


def evaluate_hgb_candidate(
    candidate_id,
    params,
    target_mode="log",
    sample_weight=None,
    use_target_encoding=False,
    te_smoothing=30.0,
):
    target_values = get_target_values(target_mode)
    oof = np.zeros(len(y), dtype=float)
    test_fold_predictions = []

    for fold_id, (train_idx, valid_idx) in enumerate(final_splits, start=1):
        pipe = build_hgb_pipeline(
            dict(params),
            use_target_encoding=use_target_encoding,
            te_smoothing=te_smoothing,
        )

        fit_kwargs = {}
        if sample_weight is not None:
            fit_kwargs["model__sample_weight"] = sample_weight[train_idx]

        try:
            pipe.fit(X_train_fe.iloc[train_idx], target_values[train_idx], **fit_kwargs)
        except TypeError:
            pipe.fit(X_train_fe.iloc[train_idx], target_values[train_idx])

        valid_pred = inverse_transform(pipe.predict(X_train_fe.iloc[valid_idx]), target_mode)
        test_pred = inverse_transform(pipe.predict(X_test_fe), target_mode)

        oof[valid_idx] = np.clip(valid_pred, 1e-9, None)
        test_fold_predictions.append(np.clip(test_pred, 1e-9, None))

    test_pred = np.mean(np.vstack(test_fold_predictions), axis=0)
    scores = regression_scores(y, oof)
    fold_mean, fold_std = fold_mape_std(oof, final_splits)

    pred_stats = {
        "prediction_min": float(np.min(test_pred)),
        "prediction_max": float(np.max(test_pred)),
        "prediction_mean": float(np.mean(test_pred)),
        "prediction_std": float(np.std(test_pred)),
    }

    row = {
        "candidate_id": candidate_id,
        "candidate_type": "hgb_single_te" if use_target_encoding else "hgb_single",
        "target_mode": target_mode,
        "members": candidate_id,
        "use_target_encoding": bool(use_target_encoding),
        "te_smoothing": te_smoothing if use_target_encoding else np.nan,
        "fold_mape_mean": fold_mean,
        "fold_mape_std": fold_std,
        **scores,
        **pred_stats,
    }

    return row, oof, test_pred


# These settings compare nearby HGB hyperparameters, target transformations, and sample weight choices.
hgb_configs = [
    {
        "candidate_id": "HGB_LOG_D6_LR035_L2_03_UNWEIGHTED",
        "target_mode": "log",
        "weights": None,
        "params": dict(
            loss="squared_error",
            max_iter=900,
            learning_rate=0.035,
            max_leaf_nodes=31,
            max_depth=6,
            min_samples_leaf=18,
            l2_regularization=0.03,
            max_bins=255,
        ),
    },
    {
        "candidate_id": "HGB_LOG_D6_LR035_L2_03_SQRT_WEIGHTED",
        "target_mode": "log",
        "weights": sqrt_weights,
        "params": dict(
            loss="squared_error",
            max_iter=900,
            learning_rate=0.035,
            max_leaf_nodes=31,
            max_depth=6,
            min_samples_leaf=18,
            l2_regularization=0.03,
            max_bins=255,
        ),
    },
    {
        "candidate_id": "HGB_LOG_D6_LR025_L2_05_UNWEIGHTED",
        "target_mode": "log",
        "weights": None,
        "params": dict(
            loss="squared_error",
            max_iter=1200,
            learning_rate=0.025,
            max_leaf_nodes=45,
            max_depth=6,
            min_samples_leaf=16,
            l2_regularization=0.05,
            max_bins=255,
        ),
    },
    {
        "candidate_id": "HGB_LOG_D6_LR025_L2_05_SQRT_WEIGHTED",
        "target_mode": "log",
        "weights": sqrt_weights,
        "params": dict(
            loss="squared_error",
            max_iter=1200,
            learning_rate=0.025,
            max_leaf_nodes=45,
            max_depth=6,
            min_samples_leaf=16,
            l2_regularization=0.05,
            max_bins=255,
        ),
    },
    {
        "candidate_id": "HGB_LOG_D7_LR020_L2_08_UNWEIGHTED",
        "target_mode": "log",
        "weights": None,
        "params": dict(
            loss="squared_error",
            max_iter=1400,
            learning_rate=0.020,
            max_leaf_nodes=63,
            max_depth=7,
            min_samples_leaf=14,
            l2_regularization=0.08,
            max_bins=255,
        ),
    },
    {
        "candidate_id": "HGB_LOG_D7_LR020_L2_08_SQRT_WEIGHTED",
        "target_mode": "log",
        "weights": sqrt_weights,
        "params": dict(
            loss="squared_error",
            max_iter=1400,
            learning_rate=0.020,
            max_leaf_nodes=63,
            max_depth=7,
            min_samples_leaf=14,
            l2_regularization=0.08,
            max_bins=255,
        ),
    },
    {
        "candidate_id": "HGB_LOG1P_D6_LR030_L2_04_SQRT_WEIGHTED",
        "target_mode": "log1p",
        "weights": sqrt_weights,
        "params": dict(
            loss="squared_error",
            max_iter=1100,
            learning_rate=0.030,
            max_leaf_nodes=45,
            max_depth=6,
            min_samples_leaf=16,
            l2_regularization=0.04,
            max_bins=255,
        ),
    },
    {
        "candidate_id": "HGB_POISSON_RAW_D6_LR030_L2_04_SQRT_WEIGHTED",
        "target_mode": "raw",
        "weights": sqrt_weights,
        "params": dict(
            loss="poisson",
            max_iter=1100,
            learning_rate=0.030,
            max_leaf_nodes=45,
            max_depth=6,
            min_samples_leaf=16,
            l2_regularization=0.04,
            max_bins=255,
        ),
    },
    {
        "candidate_id": "HGB_LOG_D6_LR030_L2_02_MAPE_WEIGHTED",
        "target_mode": "log",
        "weights": mape_weights,
        "params": dict(
            loss="squared_error",
            max_iter=1000,
            learning_rate=0.030,
            max_leaf_nodes=45,
            max_depth=6,
            min_samples_leaf=12,
            l2_regularization=0.02,
            max_bins=255,
        ),
    },
    {
        "candidate_id": "HGB_TE_LOG_D6_LR030_L2_04_SQRT_S25",
        "target_mode": "log",
        "weights": sqrt_weights,
        "use_target_encoding": True,
        "te_smoothing": 25.0,
        "params": dict(
            loss="squared_error",
            max_iter=1200,
            learning_rate=0.030,
            max_leaf_nodes=45,
            max_depth=6,
            min_samples_leaf=16,
            l2_regularization=0.04,
            max_bins=255,
        ),
    },
    {
        "candidate_id": "HGB_TE_LOG_D7_LR022_L2_08_SQRT_S30",
        "target_mode": "log",
        "weights": sqrt_weights,
        "use_target_encoding": True,
        "te_smoothing": 30.0,
        "params": dict(
            loss="squared_error",
            max_iter=1400,
            learning_rate=0.022,
            max_leaf_nodes=63,
            max_depth=7,
            min_samples_leaf=14,
            l2_regularization=0.08,
            max_bins=255,
        ),
    },
    {
        "candidate_id": "HGB_TE_LOG_D6_LR025_L2_05_UNWEIGHTED_S35",
        "target_mode": "log",
        "weights": None,
        "use_target_encoding": True,
        "te_smoothing": 35.0,
        "params": dict(
            loss="squared_error",
            max_iter=1300,
            learning_rate=0.025,
            max_leaf_nodes=45,
            max_depth=6,
            min_samples_leaf=16,
            l2_regularization=0.05,
            max_bins=255,
        ),
    },
    {
        "candidate_id": "HGB_TE_LOG_D7_LR018_L2_12_SOFT_S40",
        "target_mode": "log",
        "weights": soft_low_pressure_weights,
        "use_target_encoding": True,
        "te_smoothing": 40.0,
        "params": dict(
            loss="squared_error",
            max_iter=1600,
            learning_rate=0.018,
            max_leaf_nodes=63,
            max_depth=7,
            min_samples_leaf=12,
            l2_regularization=0.12,
            max_bins=255,
        ),
    },
]

# Additional random-state settings are tested to check model stability.
for seed in [7, 21, 42, 100, 2026]:
    hgb_configs.append({
        "candidate_id": f"HGB_LOCAL_LOG_D6_LR028_L2_04_SQRT_SEED_{seed}",
        "target_mode": "log",
        "weights": sqrt_weights,
        "params": dict(
            loss="squared_error",
            max_iter=1250,
            learning_rate=0.028,
            max_leaf_nodes=45,
            max_depth=6,
            min_samples_leaf=15,
            l2_regularization=0.04,
            max_bins=255,
            random_state=seed,
        ),
    })

for seed in [7, 42, 2026]:
    hgb_configs.append({
        "candidate_id": f"HGB_LOCAL_LOG_D7_LR021_L2_075_SQRT_SEED_{seed}",
        "target_mode": "log",
        "weights": sqrt_weights,
        "params": dict(
            loss="squared_error",
            max_iter=1450,
            learning_rate=0.021,
            max_leaf_nodes=63,
            max_depth=7,
            min_samples_leaf=13,
            l2_regularization=0.075,
            max_bins=255,
            random_state=seed,
        ),
    })

for seed in [21, 42, 100]:
    hgb_configs.append({
        "candidate_id": f"HGB_LOCAL_TE_LOG_D6_LR027_L2_045_SQRT_S30_SEED_{seed}",
        "target_mode": "log",
        "weights": sqrt_weights,
        "use_target_encoding": True,
        "te_smoothing": 30.0,
        "params": dict(
            loss="squared_error",
            max_iter=1300,
            learning_rate=0.027,
            max_leaf_nodes=45,
            max_depth=6,
            min_samples_leaf=15,
            l2_regularization=0.045,
            max_bins=255,
            random_state=seed,
        ),
    })

hgb_rows = []
hgb_oof = {}
hgb_test_predictions = {}

def _evaluate_hgb_config_safe(cfg):
    candidate_id = cfg["candidate_id"]

    if threadpool_limits is None:
        row, oof_pred, test_pred = evaluate_hgb_candidate(
            candidate_id,
            cfg["params"],
            target_mode=cfg.get("target_mode", "log"),
            sample_weight=cfg.get("weights"),
            use_target_encoding=cfg.get("use_target_encoding", False),
            te_smoothing=cfg.get("te_smoothing", 30.0),
        )
    else:
        with threadpool_limits(limits=CPU_INNER_THREADS):
            row, oof_pred, test_pred = evaluate_hgb_candidate(
                candidate_id,
                cfg["params"],
                target_mode=cfg.get("target_mode", "log"),
                sample_weight=cfg.get("weights"),
                use_target_encoding=cfg.get("use_target_encoding", False),
                te_smoothing=cfg.get("te_smoothing", 30.0),
            )

    return candidate_id, row, oof_pred, test_pred

if USE_PARALLEL_CANDIDATE_EVAL and len(hgb_configs) > 1:
    try:
        print(f"Training HGB candidates with parallel jobs: {PARALLEL_JOBS}")
        hgb_parallel_results = Parallel(
            n_jobs=PARALLEL_JOBS,
            verbose=10,
            prefer=PARALLEL_BACKEND_PREFER,
        )(
            delayed(_evaluate_hgb_config_safe)(cfg) for cfg in hgb_configs
        )
    except Exception as exc:
        print("Parallel HGB training was not available. Running candidates one at a time:", repr(exc))
        hgb_parallel_results = []

        for cfg in hgb_configs:
            print("Training HGB candidate:", cfg["candidate_id"])
            hgb_parallel_results.append(_evaluate_hgb_config_safe(cfg))
else:
    hgb_parallel_results = []

    for cfg in hgb_configs:
        print("Training HGB candidate:", cfg["candidate_id"])
        hgb_parallel_results.append(_evaluate_hgb_config_safe(cfg))

for candidate_id, row, oof_pred, test_pred in hgb_parallel_results:
    hgb_rows.append(row)
    hgb_oof[candidate_id] = oof_pred
    hgb_test_predictions[candidate_id] = test_pred

base_hgb_results = (
    pd.DataFrame(hgb_rows)
    .sort_values(["mape", "fold_mape_std"])
    .reset_index(drop=True)
)

display(base_hgb_results)

In [ ]:
def add_hgb_average(candidate_id, members, weights=None, candidate_type="hgb_average"):
    members = [m for m in members if m in hgb_oof]
    if not members:
        return

    oof_matrix = np.vstack([hgb_oof[m] for m in members])
    test_matrix = np.vstack([hgb_test_predictions[m] for m in members])

    if weights is None:
        oof_pred = np.mean(oof_matrix, axis=0)
        test_pred = np.mean(test_matrix, axis=0)
    else:
        weights = np.asarray(weights, dtype=float)
        weights = weights / weights.sum()
        oof_pred = np.average(oof_matrix, axis=0, weights=weights)
        test_pred = np.average(test_matrix, axis=0, weights=weights)

    hgb_oof[candidate_id] = np.clip(oof_pred, 1e-9, None)
    hgb_test_predictions[candidate_id] = np.clip(test_pred, 1e-9, None)

    scores = regression_scores(y, hgb_oof[candidate_id])
    fold_mean, fold_std = fold_mape_std(hgb_oof[candidate_id], final_splits)

    hgb_rows.append({
        "candidate_id": candidate_id,
        "candidate_type": candidate_type,
        "target_mode": "mixed_hgb",
        "members": " | ".join(members),
        "fold_mape_mean": fold_mean,
        "fold_mape_std": fold_std,
        **scores,
        "prediction_min": float(np.min(hgb_test_predictions[candidate_id])),
        "prediction_max": float(np.max(hgb_test_predictions[candidate_id])),
        "prediction_mean": float(np.mean(hgb_test_predictions[candidate_id])),
        "prediction_std": float(np.std(hgb_test_predictions[candidate_id])),
    })


def passes_prediction_range_check(row):
    return bool(
        np.isfinite([
            row["prediction_min"],
            row["prediction_max"],
            row["prediction_mean"],
            row["prediction_std"],
        ]).all()
        and row["prediction_min"] > 0
        and row["prediction_max"] < 12.0
        and 0.10 < row["prediction_mean"] < 0.80
        and 0.08 < row["prediction_std"] < 1.50
    )


ordered_base = base_hgb_results.sort_values(["mape", "fold_mape_std"])["candidate_id"].tolist()

for top_k in [2, 3, 4, 5]:
    if len(ordered_base) >= top_k:
        selected = ordered_base[:top_k]
        add_hgb_average(f"HGB_MEAN_TOP{top_k}", selected, candidate_type="hgb_mean")

        local_mape = base_hgb_results.set_index("candidate_id").loc[selected]["mape"].values
        relative = (local_mape - local_mape.min()) / max(local_mape.min(), 1e-9)
        soft_weights = np.exp(-35.0 * relative)

        add_hgb_average(
            f"HGB_SOFT_TOP{top_k}",
            selected,
            weights=soft_weights,
            candidate_type="hgb_soft_weighted",
        )

# This candidate combines overall validation performance with low-pressure coverage.
base_with_low = pd.DataFrame(hgb_rows).copy()
low_candidates = base_with_low.sort_values(["prediction_min", "mape"]).head(4)["candidate_id"].tolist()
best_overall = ordered_base[0] if ordered_base else None

if best_overall and low_candidates:
    low_member = low_candidates[0]
    add_hgb_average(
        "HGB_LOW_PRESSURE_AWARE_BLEND",
        [best_overall, low_member],
        weights=[0.75, 0.25],
        candidate_type="hgb_low_pressure_blend",
    )

hgb_final_family_results = pd.DataFrame(hgb_rows)
hgb_final_family_results["prediction_range_check"] = hgb_final_family_results.apply(
    passes_prediction_range_check,
    axis=1,
)
hgb_final_family_results["eligible_for_final"] = hgb_final_family_results["prediction_range_check"]

# Validation MAPE is the main selection signal. Fold variation and R2 are used as secondary tie-breakers.
hgb_final_family_results["decision_score"] = (
    hgb_final_family_results["mape"]
    + 0.50 * hgb_final_family_results["fold_mape_std"]
    - 0.005 * hgb_final_family_results["r2"]
)

hgb_final_family_results = hgb_final_family_results.sort_values(
    ["eligible_for_final", "decision_score", "mape"],
    ascending=[False, True, True],
).reset_index(drop=True)

display(hgb_final_family_results)
hgb_final_family_results.to_csv(TABLE_DIR / "03_hgb_model_tuning_summary.csv", index=False)

plt.figure(figsize=(11, 5))
plot_df = hgb_final_family_results.head(12).copy()
plt.bar(plot_df["candidate_id"], plot_df["mape"])
plt.xticks(rotation=75, ha="right")
plt.ylabel("Validation MAPE")
plt.title("HGB Candidate Models by MAPE")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "hgb_candidate_mape.png", dpi=150)
plt.show()



## 8. HistGradientBoosting and MLP blend search

The final ensemble selection compares the HistGradientBoosting predictions with MLP support predictions.

HistGradientBoosting is used as the main tabular model component because it performed strongly during validation. MLP is included as a support model to provide an additional nonlinear prediction signal.

This comparison table also keeps HGB only candidates so their validation performance can be compared with the blended candidates.

In [ ]:
def evaluate_mlp_candidate(
    candidate_id,
    hidden_layers,
    alpha,
    learning_rate_init,
    max_iter=600,
    use_target_encoding=False,
    te_smoothing=35.0,
    seed=RANDOM_STATE,
):
    model = MLPRegressor(
        hidden_layer_sizes=hidden_layers,
        alpha=alpha,
        learning_rate_init=learning_rate_init,
        activation="relu",
        solver="adam",
        batch_size=128,
        max_iter=max_iter,
        early_stopping=True,
        validation_fraction=0.12,
        n_iter_no_change=35,
        random_state=seed,
    )

    if use_target_encoding:
        pipe = Pipeline([
            ("target_encoding", SmoothedTargetEncoderFeatures(
                columns=TARGET_ENCODING_COLUMNS,
                smoothing=te_smoothing,
                prefix="TE_",
            )),
            ("preprocessor", make_dynamic_scaled_preprocessor()),
            ("model", model),
        ])
    else:
        pipe = Pipeline([
            ("preprocessor", make_scaled_preprocessor()),
            ("model", model),
        ])

    oof = np.zeros(len(y), dtype=float)
    test_fold_predictions = []

    for train_idx, valid_idx in final_splits:
        fitted = clone(pipe)
        fitted.fit(X_train_fe.iloc[train_idx], log_y[train_idx])

        valid_pred = np.exp(fitted.predict(X_train_fe.iloc[valid_idx]))
        test_pred = np.exp(fitted.predict(X_test_fe))

        oof[valid_idx] = np.clip(valid_pred, 1e-9, None)
        test_fold_predictions.append(np.clip(test_pred, 1e-9, None))

    test_pred = np.mean(np.vstack(test_fold_predictions), axis=0)
    scores = regression_scores(y, oof)
    fold_mean, fold_std = fold_mape_std(oof, final_splits)

    row = {
        "candidate_id": candidate_id,
        "candidate_type": "mlp_support_te" if use_target_encoding else "mlp_support",
        "target_mode": "log",
        "members": candidate_id,
        "use_target_encoding": bool(use_target_encoding),
        "te_smoothing": te_smoothing if use_target_encoding else np.nan,
        "fold_mape_mean": fold_mean,
        "fold_mape_std": fold_std,
        **scores,
        "prediction_min": float(np.min(test_pred)),
        "prediction_max": float(np.max(test_pred)),
        "prediction_mean": float(np.mean(test_pred)),
        "prediction_std": float(np.std(test_pred)),
    }

    return row, oof, test_pred


# MLP models are trained as support models for comparison in the final ensemble search.
mlp_settings = []

for seed in [7, 21, 42, 100, 2026]:
    mlp_settings.append((
        f"MLP_SUPPORT_B_SEED_{seed}",
        (128, 64),
        0.0040,
        0.0012,
        False,
        30.0,
        seed,
    ))

for seed in [7, 42, 2026]:
    mlp_settings.append((
        f"MLP_SUPPORT_D_SEED_{seed}",
        (144, 72),
        0.0060,
        0.0010,
        False,
        30.0,
        seed,
    ))

for seed in [21, 42]:
    mlp_settings.append((
        f"MLP_SUPPORT_B_FINE_SEED_{seed}",
        (128, 64),
        0.0035,
        0.0011,
        False,
        30.0,
        seed,
    ))

for seed in [42, 2026]:
    mlp_settings.append((
        f"MLP_SUPPORT_TE_E_SEED_{seed}",
        (128, 64),
        0.0060,
        0.0011,
        True,
        35.0,
        seed,
    ))

mlp_rows = []
mlp_oof = {}
mlp_test_predictions = {}


def evaluate_mlp_model(settings):
    candidate_id, hidden_layers, alpha, lr, use_te, te_smoothing, seed = settings

    if threadpool_limits is None:
        row, oof_pred, test_pred = evaluate_mlp_candidate(
            candidate_id,
            hidden_layers,
            alpha,
            lr,
            use_target_encoding=use_te,
            te_smoothing=te_smoothing,
            seed=seed,
        )
    else:
        with threadpool_limits(limits=CPU_INNER_THREADS):
            row, oof_pred, test_pred = evaluate_mlp_candidate(
                candidate_id,
                hidden_layers,
                alpha,
                lr,
                use_target_encoding=use_te,
                te_smoothing=te_smoothing,
                seed=seed,
            )

    return candidate_id, row, oof_pred, test_pred


MLP_PARALLEL_JOBS = max(1, min(4, PARALLEL_JOBS))

if USE_PARALLEL_CANDIDATE_EVAL and len(mlp_settings) > 1:
    try:
        print(f"Training MLP support models with parallel jobs: {MLP_PARALLEL_JOBS}")
        mlp_parallel_results = Parallel(
            n_jobs=MLP_PARALLEL_JOBS,
            verbose=10,
            prefer=PARALLEL_BACKEND_PREFER,
        )(
            delayed(evaluate_mlp_model)(settings) for settings in mlp_settings
        )
    except Exception as exc:
        print("Parallel MLP training was not available. Running models one at a time:", repr(exc))
        mlp_parallel_results = []

        for settings in mlp_settings:
            print("Training MLP support model:", settings[0])
            mlp_parallel_results.append(evaluate_mlp_model(settings))
else:
    mlp_parallel_results = []

    for settings in mlp_settings:
        print("Training MLP support model:", settings[0])
        mlp_parallel_results.append(evaluate_mlp_model(settings))

for candidate_id, row, oof_pred, test_pred in mlp_parallel_results:
    mlp_rows.append(row)
    mlp_oof[candidate_id] = oof_pred
    mlp_test_predictions[candidate_id] = test_pred


def add_mlp_average(candidate_id, members, weights=None, candidate_type="mlp_seed_average"):
    members = [m for m in members if m in mlp_oof]
    if not members:
        return

    oof_matrix = np.vstack([mlp_oof[m] for m in members])
    test_matrix = np.vstack([mlp_test_predictions[m] for m in members])

    if weights is None:
        oof_pred = np.mean(oof_matrix, axis=0)
        test_pred = np.mean(test_matrix, axis=0)
    else:
        weights = np.asarray(weights, dtype=float)
        weights = weights / weights.sum()
        oof_pred = np.average(oof_matrix, axis=0, weights=weights)
        test_pred = np.average(test_matrix, axis=0, weights=weights)

    mlp_oof[candidate_id] = np.clip(oof_pred, 1e-9, None)
    mlp_test_predictions[candidate_id] = np.clip(test_pred, 1e-9, None)

    scores = regression_scores(y, mlp_oof[candidate_id])
    fold_mean, fold_std = fold_mape_std(mlp_oof[candidate_id], final_splits)

    mlp_rows.append({
        "candidate_id": candidate_id,
        "candidate_type": candidate_type,
        "target_mode": "log",
        "members": " | ".join(members),
        "use_target_encoding": False,
        "te_smoothing": np.nan,
        "fold_mape_mean": fold_mean,
        "fold_mape_std": fold_std,
        **scores,
        "prediction_min": float(np.min(mlp_test_predictions[candidate_id])),
        "prediction_max": float(np.max(mlp_test_predictions[candidate_id])),
        "prediction_mean": float(np.mean(mlp_test_predictions[candidate_id])),
        "prediction_std": float(np.std(mlp_test_predictions[candidate_id])),
    })


add_mlp_average(
    "MLP_SUPPORT_B_SEED_AVERAGE",
    [f"MLP_SUPPORT_B_SEED_{s}" for s in [7, 21, 42, 100, 2026]],
)

add_mlp_average(
    "MLP_SUPPORT_D_SEED_AVERAGE",
    [f"MLP_SUPPORT_D_SEED_{s}" for s in [7, 42, 2026]],
)

add_mlp_average(
    "MLP_SUPPORT_BD_AVERAGE",
    ["MLP_SUPPORT_B_SEED_AVERAGE", "MLP_SUPPORT_D_SEED_AVERAGE"],
    weights=[0.65, 0.35],
)

mlp_branch_results = pd.DataFrame(mlp_rows)
mlp_branch_results["prediction_range_check"] = mlp_branch_results.apply(
    passes_prediction_range_check,
    axis=1,
)
mlp_branch_results["decision_score"] = (
    mlp_branch_results["mape"]
    + 0.60 * mlp_branch_results["fold_mape_std"]
    - 0.005 * mlp_branch_results["r2"]
)
mlp_branch_results = (
    mlp_branch_results
    .sort_values(["decision_score", "mape"])
    .reset_index(drop=True)
)

display(mlp_branch_results)
mlp_branch_results.to_csv(TABLE_DIR / "03b_mlp_support_summary.csv", index=False)


def passes_blend_range_check(row):
    return passes_prediction_range_check(row)


y_array_for_deciles = np.asarray(y, dtype=float)
low_target_cut = np.quantile(y_array_for_deciles, 0.10)
low_target_mask = y_array_for_deciles <= low_target_cut


# Low-pressure validation is checked because MAPE is sensitive when target values are small.
def low_decile_mape_score(pred):
    pred = np.asarray(pred, dtype=float)
    return float(mean_absolute_percentage_error(
        y_array_for_deciles[low_target_mask],
        np.clip(pred[low_target_mask], 1e-9, None),
    ))


def candidate_row_from_predictions(candidate_id, candidate_type, members, oof_pred, test_pred):
    scores = regression_scores(y, oof_pred)
    fold_mean, fold_std = fold_mape_std(oof_pred, final_splits)

    row = {
        "candidate_id": candidate_id,
        "candidate_type": candidate_type,
        "target_mode": "mixed_log",
        "members": " | ".join(members),
        "fold_mape_mean": fold_mean,
        "fold_mape_std": fold_std,
        **scores,
        "prediction_min": float(np.min(test_pred)),
        "prediction_max": float(np.max(test_pred)),
        "prediction_mean": float(np.mean(test_pred)),
        "prediction_std": float(np.std(test_pred)),
        "low_decile_mape": low_decile_mape_score(oof_pred),
    }

    row["prediction_range_check"] = passes_blend_range_check(row)
    row["eligible_for_final"] = row["prediction_range_check"]
    row["decision_score"] = (
        row["mape"]
        + 0.60 * row["fold_mape_std"]
        - 0.005 * row["r2"]
    )

    return row


def weighted_raw_average(prediction_map, weights):
    total = None

    for key, weight in weights:
        arr = np.clip(prediction_map[key], 1e-9, None)
        total = weight * arr if total is None else total + weight * arr

    return np.clip(total, 1e-9, None)


def weighted_log_average(prediction_map, weights):
    total = None

    for key, weight in weights:
        arr = np.log(np.clip(prediction_map[key], 1e-9, None))
        total = weight * arr if total is None else total + weight * arr

    return np.clip(np.exp(total), 1e-9, None)


blend_rows = []
blend_oof = {}
blend_test_predictions = {}

candidate_oof = dict(hgb_oof)
candidate_test = dict(hgb_test_predictions)
candidate_oof.update(mlp_oof)
candidate_test.update(mlp_test_predictions)

hgb_candidate_table = hgb_final_family_results.copy()
hgb_candidate_table["prediction_range_check"] = hgb_candidate_table.apply(
    passes_blend_range_check,
    axis=1,
)

eligible_hgb_pool = (
    hgb_candidate_table[hgb_candidate_table["prediction_range_check"]]
    .sort_values(["decision_score", "mape"])["candidate_id"]
    .head(12)
    .tolist()
)

if not eligible_hgb_pool:
    eligible_hgb_pool = (
        hgb_candidate_table[hgb_candidate_table["eligible_for_final"]]
        .sort_values(["decision_score", "mape"])["candidate_id"]
        .head(12)
        .tolist()
    )

mlp_pool = (
    mlp_branch_results
    .sort_values(["decision_score", "mape"])["candidate_id"]
    .head(8)
    .tolist()
)

stable_hgb_average_id = "HGB_MEAN_TOP4"
stable_mlp_average_id = "MLP_SUPPORT_B_SEED_AVERAGE"

if stable_hgb_average_id in hgb_oof and stable_hgb_average_id not in eligible_hgb_pool:
    eligible_hgb_pool.append(stable_hgb_average_id)

if stable_mlp_average_id in mlp_oof and stable_mlp_average_id not in mlp_pool:
    mlp_pool.append(stable_mlp_average_id)

eligible_hgb_pool = list(dict.fromkeys(eligible_hgb_pool))
mlp_pool = list(dict.fromkeys(mlp_pool))

print("HGB models selected for blending:", len(eligible_hgb_pool))
print("MLP support models selected for blending:", len(mlp_pool))

for hgb_id in eligible_hgb_pool:
    row = candidate_row_from_predictions(
        hgb_id,
        "hgb_candidate",
        [hgb_id],
        candidate_oof[hgb_id],
        candidate_test[hgb_id],
    )
    blend_rows.append(row)

for hgb_id in eligible_hgb_pool[:10]:
    for mlp_id in mlp_pool:
        for mlp_weight in [0.20, 0.21, 0.22, 0.23, 0.24, 0.25, 0.26, 0.27, 0.28]:
            hgb_weight = 1.0 - mlp_weight
            weights = [(hgb_id, hgb_weight), (mlp_id, mlp_weight)]

            oof_pred = weighted_raw_average(candidate_oof, weights)
            test_pred = weighted_raw_average(candidate_test, weights)

            candidate_id = (
                f"BLEND_RAW_{hgb_id}_{mlp_id}_"
                f"HGB{int(round(hgb_weight * 100)):02d}_"
                f"MLP{int(round(mlp_weight * 100)):02d}"
            )

            row = candidate_row_from_predictions(
                candidate_id,
                "hgb_mlp_raw_blend",
                [f"{hgb_id}:{hgb_weight:.2f}", f"{mlp_id}:{mlp_weight:.2f}"],
                oof_pred,
                test_pred,
            )

            blend_rows.append(row)
            blend_oof[candidate_id] = oof_pred
            blend_test_predictions[candidate_id] = test_pred

for hgb_id in eligible_hgb_pool[:8]:
    for mlp_id in mlp_pool[:6]:
        for mlp_weight in [0.18, 0.20, 0.22, 0.24, 0.25, 0.26]:
            hgb_weight = 1.0 - mlp_weight
            weights = [(hgb_id, hgb_weight), (mlp_id, mlp_weight)]

            oof_pred = weighted_log_average(candidate_oof, weights)
            test_pred = weighted_log_average(candidate_test, weights)

            candidate_id = (
                f"BLEND_LOG_{hgb_id}_{mlp_id}_"
                f"HGB{int(round(hgb_weight * 100)):02d}_"
                f"MLP{int(round(mlp_weight * 100)):02d}"
            )

            row = candidate_row_from_predictions(
                candidate_id,
                "hgb_mlp_log_blend",
                [f"{hgb_id}:{hgb_weight:.2f}", f"{mlp_id}:{mlp_weight:.2f}"],
                oof_pred,
                test_pred,
            )

            blend_rows.append(row)
            blend_oof[candidate_id] = oof_pred
            blend_test_predictions[candidate_id] = test_pred

if len(eligible_hgb_pool) >= 2:
    hgb_pair_candidates = [(eligible_hgb_pool[0], eligible_hgb_pool[1])]

    if len(eligible_hgb_pool) >= 4:
        hgb_pair_candidates.append((eligible_hgb_pool[0], eligible_hgb_pool[3]))

    for h1, h2 in hgb_pair_candidates:
        for mlp_id in mlp_pool[:6]:
            for mlp_weight in [0.20, 0.22, 0.24, 0.25, 0.26]:
                hgb_total = 1.0 - mlp_weight
                weights = [
                    (h1, hgb_total * 0.65),
                    (h2, hgb_total * 0.35),
                    (mlp_id, mlp_weight),
                ]

                for blend_mode, blend_function in [
                    ("RAW", weighted_raw_average),
                    ("LOG", weighted_log_average),
                ]:
                    oof_pred = blend_function(candidate_oof, weights)
                    test_pred = blend_function(candidate_test, weights)

                    candidate_id = (
                        f"BLEND_{blend_mode}_HGBPAIR_{h1}_{h2}_{mlp_id}_"
                        f"MLP{int(round(mlp_weight * 100)):02d}"
                    )

                    row = candidate_row_from_predictions(
                        candidate_id,
                        f"hgb_pair_mlp_{blend_mode.lower()}_blend",
                        [
                            f"{h1}:{hgb_total * 0.65:.2f}",
                            f"{h2}:{hgb_total * 0.35:.2f}",
                            f"{mlp_id}:{mlp_weight:.2f}",
                        ],
                        oof_pred,
                        test_pred,
                    )

                    blend_rows.append(row)
                    blend_oof[candidate_id] = oof_pred
                    blend_test_predictions[candidate_id] = test_pred

for hgb_id in eligible_hgb_pool[:6]:
    for mlp_id in mlp_pool[:6]:
        for hgb_weight, mlp_weight in [
            (0.78, 0.22),
            (0.77, 0.23),
            (0.76, 0.24),
            (0.75, 0.25),
            (0.74, 0.26),
        ]:
            weights = [(hgb_id, hgb_weight), (mlp_id, mlp_weight)]

            for blend_mode, blend_function in [
                ("RAW_LOCAL", weighted_raw_average),
                ("LOG_LOCAL", weighted_log_average),
            ]:
                oof_pred = blend_function(candidate_oof, weights)
                test_pred = blend_function(candidate_test, weights)

                candidate_id = (
                    f"BLEND_{blend_mode}_{hgb_id}_{mlp_id}_"
                    f"HGB{int(round(hgb_weight * 100)):02d}_"
                    f"MLP{int(round(mlp_weight * 100)):02d}"
                )

                row = candidate_row_from_predictions(
                    candidate_id,
                    f"hgb_mlp_{blend_mode.lower()}_blend",
                    [f"{hgb_id}:{hgb_weight:.2f}", f"{mlp_id}:{mlp_weight:.2f}"],
                    oof_pred,
                    test_pred,
                )

                blend_rows.append(row)
                blend_oof[candidate_id] = oof_pred
                blend_test_predictions[candidate_id] = test_pred

blend_search_results = (
    pd.DataFrame(blend_rows)
    .sort_values(["eligible_for_final", "decision_score", "mape"], ascending=[False, True, True])
    .reset_index(drop=True)
)

display(blend_search_results.head(40))
blend_search_results.to_csv(TABLE_DIR / "03c_blend_selection_summary.csv", index=False)

hgb_oof.update(mlp_oof)
hgb_test_predictions.update(mlp_test_predictions)
hgb_oof.update(blend_oof)
hgb_test_predictions.update(blend_test_predictions)

### Hyperparameter tuning summary

This table summarises the main hyperparameters considered for each model family and the final settings used in the notebook.

In [ ]:
hyperparameter_summary = pd.DataFrame([
    {
        "model_family": "Regularised Linear",
        "model": "Ridge",
        "hyperparameters_reviewed": "alpha",
        "values_considered": "10.0",
        "selected_values": "alpha = 10.0",
        "role_in_notebook": "Baseline comparison model",
    },
    {
        "model_family": "Regularised Linear",
        "model": "ElasticNet",
        "hyperparameters_reviewed": "alpha, l1_ratio, max_iter",
        "values_considered": "alpha = 0.002, l1_ratio = 0.15, max_iter = 6000",
        "selected_values": "alpha = 0.002, l1_ratio = 0.15, max_iter = 6000",
        "role_in_notebook": "Baseline comparison model",
    },
    {
        "model_family": "Support Vector Regression",
        "model": "LinearSVR",
        "hyperparameters_reviewed": "C, epsilon, max_iter",
        "values_considered": "C = 1.0, epsilon = 0.02, max_iter = 5000",
        "selected_values": "C = 1.0, epsilon = 0.02, max_iter = 5000",
        "role_in_notebook": "Comparison model",
    },
    {
        "model_family": "Neural Network",
        "model": "MLPRegressor",
        "hyperparameters_reviewed": "hidden_layer_sizes, alpha, learning_rate_init, random_state",
        "values_considered": "(128, 64), (144, 72); alpha = 0.0035 to 0.0060; learning_rate_init = 0.0010 to 0.0012; several random seeds",
        "selected_values": "MLP support averages based on B and D seed groups",
        "role_in_notebook": "Support model for final ensemble comparison",
    },
    {
        "model_family": "Tree Ensemble",
        "model": "HistGradientBoostingRegressor",
        "hyperparameters_reviewed": "loss, max_iter, learning_rate, max_leaf_nodes, max_depth, min_samples_leaf, l2_regularization, max_bins",
        "values_considered": "learning_rate = 0.018 to 0.035; max_iter = 900 to 1600; max_leaf_nodes = 31 to 63; max_depth = 6 to 7; l2_regularization = 0.02 to 0.12",
        "selected_values": "HGB model components selected by cross-validation and combined in HGB_MEAN_TOP3",
        "role_in_notebook": "Main tabular model component",
    },
    {
        "model_family": "Ensemble",
        "model": "HGB and MLP blend",
        "hyperparameters_reviewed": "HGB weight, MLP weight, raw average, log average",
        "values_considered": "MLP weight = 0.18 to 0.28 with raw and log prediction averaging",
        "selected_values": "HGB-dominant blend selected by validation evidence",
        "role_in_notebook": "Final ensemble base",
    },
    {
        "model_family": "Residual Calibration",
        "model": "HuberRegressor",
        "hyperparameters_reviewed": "epsilon, alpha, shrinkage, cap",
        "values_considered": "epsilon = 1.20; alpha = 0.0001 and 0.0005; shrinkage = 0.45; cap = 0.005",
        "selected_values": "epsilon = 1.20, alpha = 0.0001, shrinkage = 0.45, cap = 0.005",
        "role_in_notebook": "Final residual calibration",
    },
])

display(hyperparameter_summary)
hyperparameter_summary.to_csv(TABLE_DIR / "04_hyperparameter_tuning_summary.csv", index=False)

## 9. Final selected ensemble and residual calibration

This section applies the selected residual calibration setting to the cross-validated model predictions.

The final selected model uses a HistGradientBoosting-dominant ensemble with MLP support, followed by cross-validated Huber residual calibration with controlled adjustment size.

The selected model is then used to export the final `prediction.csv` file.

In [ ]:
from sklearn.linear_model import HuberRegressor
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

def write_final_table(df, filename, display_head=30):
    path = TABLE_DIR / filename
    table = pd.DataFrame(df).copy()
    table.to_csv(path, index=False)
    print("Saved table:", path)
    if display_head:
        display(table.head(display_head))
    return path

def check_prediction_array(name, values, expected_len):
    values = np.asarray(values, dtype=float)

    if len(values) != expected_len:
        raise ValueError(f"{name} length mismatch: got {len(values)}, expected {expected_len}")

    if not np.all(np.isfinite(values)):
        raise ValueError(f"{name} contains invalid numeric values.")

    if np.any(values <= 0):
        raise ValueError(f"{name} contains non-positive predictions.")

    return np.clip(values, 1e-9, None)

required_hgb_component = "HGB_MEAN_TOP3"
required_mlp_component = "MLP_SUPPORT_BD_AVERAGE"

if "hgb_oof" not in globals() or "hgb_test_predictions" not in globals():
    raise RuntimeError("Run the HGB tuning section before this cell.")

if "mlp_oof" not in globals() or "mlp_test_predictions" not in globals():
    raise RuntimeError("Run the MLP support section before this cell.")

required_components = [
    (required_hgb_component, hgb_oof, "HGB training predictions"),
    (required_hgb_component, hgb_test_predictions, "HGB test predictions"),
    (required_mlp_component, mlp_oof, "MLP training predictions"),
    (required_mlp_component, mlp_test_predictions, "MLP test predictions"),
]

missing_components = [
    f"{label}: {component}"
    for component, mapping, label in required_components
    if component not in mapping
]

if missing_components:
    raise KeyError("Missing required model components: " + ", ".join(missing_components))

final_hgb_oof = dict(hgb_oof)
final_hgb_test = dict(hgb_test_predictions)
final_mlp_oof = dict(mlp_oof)
final_mlp_test = dict(mlp_test_predictions)

y_values = np.asarray(y, dtype=float)

if len(y_values) != len(X_train_fe):
    raise RuntimeError("Target length and training feature length do not match.")

# This calibration step adjusts the base prediction using training fold prediction ratios.
# it is fitted inside each validation fold before being applied to that fold
def fit_bin_ratio_calibrator(y_train, pred_train, n_bins=8, smoothing=90.0, clip_low=0.95, clip_high=1.05):
    pred_train = np.clip(np.asarray(pred_train, dtype=float), 1e-9, None)
    ratio = np.asarray(y_train, dtype=float) / pred_train
    ratio = np.clip(ratio, 0.50, 1.80)

    try:
        _, edges = pd.qcut(pred_train, q=n_bins, retbins=True, duplicates="drop")
    except ValueError:
        edges = np.quantile(pred_train, np.linspace(0, 1, min(n_bins, len(pred_train)) + 1))

    edges = np.unique(edges)

    if len(edges) < 3:
        edges = np.array([
            float(pred_train.min()),
            float(pred_train.mean()),
            float(pred_train.max()),
        ])

    bins = np.digitize(pred_train, edges[1:-1], right=False)
    global_ratio = float(np.median(ratio))
    mapping = {}

    for bin_id in np.unique(bins):
        mask = bins == bin_id
        local_ratio = float(np.median(ratio[mask]))
        count = int(mask.sum())
        smoothed = (
            local_ratio * count + global_ratio * float(smoothing)
        ) / (count + float(smoothing))
        mapping[int(bin_id)] = float(np.clip(smoothed, clip_low, clip_high))

    default_ratio = float(np.clip(global_ratio, clip_low, clip_high))
    return edges, mapping, default_ratio

def apply_bin_ratio_calibrator(pred, edges, mapping, default_ratio):
    pred = np.clip(np.asarray(pred, dtype=float), 1e-9, None)
    bins = np.digitize(pred, edges[1:-1], right=False)
    correction = np.array([mapping.get(int(bin_id), default_ratio) for bin_id in bins], dtype=float)
    return np.clip(pred * correction, 1e-9, None)

def crossfit_bin_ratio_calibration(base_oof, base_test, n_bins=8, smoothing=90.0, clip_low=0.95, clip_high=1.05):
    base_oof = np.clip(np.asarray(base_oof, dtype=float), 1e-9, None)
    base_test = np.clip(np.asarray(base_test, dtype=float), 1e-9, None)
    calibrated_oof = np.zeros(len(y_values), dtype=float)

    for train_idx, valid_idx in final_splits:
        edges, mapping, default_ratio = fit_bin_ratio_calibrator(
            y_values[train_idx],
            base_oof[train_idx],
            n_bins=n_bins,
            smoothing=smoothing,
            clip_low=clip_low,
            clip_high=clip_high,
        )
        calibrated_oof[valid_idx] = apply_bin_ratio_calibrator(
            base_oof[valid_idx],
            edges,
            mapping,
            default_ratio,
        )

    edges, mapping, default_ratio = fit_bin_ratio_calibrator(
        y_values,
        base_oof,
        n_bins=n_bins,
        smoothing=smoothing,
        clip_low=clip_low,
        clip_high=clip_high,
    )
    calibrated_test = apply_bin_ratio_calibrator(base_test, edges, mapping, default_ratio)

    return np.clip(calibrated_oof, 1e-9, None), np.clip(calibrated_test, 1e-9, None)

base_calibration_specs = {
    "C0575": {"n_bins": 8, "smoothing": 82.50, "clip_low": 0.94250, "clip_high": 1.05750},
    "C060": {"n_bins": 8, "smoothing": 80.00, "clip_low": 0.94000, "clip_high": 1.06000},
}

def build_raw_blend(hgb_weight, hgb_component=required_hgb_component, mlp_component=required_mlp_component):
    hgb_weight = float(hgb_weight)
    oof = (
        hgb_weight * final_hgb_oof[hgb_component]
        + (1.0 - hgb_weight) * final_mlp_oof[mlp_component]
    )
    test = (
        hgb_weight * final_hgb_test[hgb_component]
        + (1.0 - hgb_weight) * final_mlp_test[mlp_component]
    )
    return np.clip(oof, 1e-9, None), np.clip(test, 1e-9, None)

def build_base_predictions(hgb_weight, base_calibration, hgb_component=required_hgb_component, mlp_component=required_mlp_component):
    raw_oof, raw_test = build_raw_blend(hgb_weight, hgb_component, mlp_component)
    return crossfit_bin_ratio_calibration(
        raw_oof,
        raw_test,
        **base_calibration_specs[base_calibration],
    )

def apply_huber_correction(base_pred, residual_pred, shrinkage, clip_low, clip_high):
    correction = np.exp(float(shrinkage) * np.asarray(residual_pred, dtype=float))
    correction = np.clip(correction, clip_low, clip_high)
    adjusted = np.asarray(base_pred, dtype=float) * correction
    return np.clip(adjusted, 1e-9, None), correction

def apply_ratio_cap(candidate_pred, base_pred, cap):
    candidate_pred = np.clip(np.asarray(candidate_pred, dtype=float), 1e-9, None)
    base_pred = np.clip(np.asarray(base_pred, dtype=float), 1e-9, None)
    ratio = candidate_pred / base_pred
    capped_ratio = np.clip(ratio, 1.0 - float(cap), 1.0 + float(cap))
    return np.clip(base_pred * capped_ratio, 1e-9, None), capped_ratio

def make_decile_bins(values, q=10):
    values = pd.Series(np.asarray(values, dtype=float))

    try:
        return (
            pd.qcut(values, q=q, labels=False, duplicates="drop")
            .astype(float)
            .fillna(0)
            .astype(int)
            .to_numpy()
        )
    except Exception:
        ranks = values.rank(method="average", pct=True).fillna(0.5).to_numpy()
        return np.clip((ranks * q).astype(int), 0, q - 1)

# Residual features describe the difference between the base ensemble and its support models
# these features are used to model remaining log scale prediction error.
def build_residual_features(base_pred, hgb_pred, mlp_pred, X_frame):
    base_pred = np.clip(np.asarray(base_pred, dtype=float), 1e-9, None)
    hgb_pred = np.clip(np.asarray(hgb_pred, dtype=float), 1e-9, None)
    mlp_pred = np.clip(np.asarray(mlp_pred, dtype=float), 1e-9, None)

    signed_difference = hgb_pred - mlp_pred
    absolute_difference = np.abs(signed_difference)
    relative_difference = signed_difference / np.maximum(base_pred, 1e-9)

    data = pd.DataFrame({
        "log_base": np.log(base_pred),
        "base_pred": base_pred,
        "hgb_pred": hgb_pred,
        "mlp_pred": mlp_pred,
        "signed_disagreement": signed_difference,
        "absolute_disagreement": absolute_difference,
        "relative_disagreement": relative_difference,
        "prediction_decile": make_decile_bins(base_pred, 10),
        "absolute_disagreement_decile": make_decile_bins(absolute_difference, 10),
        "positive_disagreement": (signed_difference > 0).astype(float),
        "negative_disagreement": (signed_difference < 0).astype(float),
    }, index=X_frame.index)

    numeric_context_columns = [
        c for c in X_frame.columns
        if pd.api.types.is_numeric_dtype(X_frame[c])
    ]

    for column in numeric_context_columns[:20]:
        values = pd.to_numeric(X_frame[column], errors="coerce")
        safe_column_name = re.sub(r"[^A-Za-z0-9]+", "_", str(column)).strip("_")[:45]
        data[f"ctx_{safe_column_name}"] = values.fillna(values.median()).to_numpy()

    return data.replace([np.inf, -np.inf], np.nan).fillna(0.0)

def fit_predict_huber_residual(base_oof, base_test, hgb_oof_values, hgb_test_values, mlp_oof_values, mlp_test_values, epsilon=1.20, alpha=0.0001):
    train_features = build_residual_features(base_oof, hgb_oof_values, mlp_oof_values, X_train_fe)
    test_features = build_residual_features(base_test, hgb_test_values, mlp_test_values, X_test_fe)

    residual_target = (
        np.log(np.clip(y_values, 1e-9, None))
        - np.log(np.clip(base_oof, 1e-9, None))
    )

    residual_oof = np.zeros(len(y_values), dtype=float)

    for train_idx, valid_idx in final_splits:
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("huber", HuberRegressor(epsilon=float(epsilon), alpha=float(alpha), max_iter=700)),
        ])
        model.fit(train_features.iloc[train_idx], residual_target[train_idx])
        residual_oof[valid_idx] = model.predict(train_features.iloc[valid_idx])

    final_model = Pipeline([
        ("scaler", StandardScaler()),
        ("huber", HuberRegressor(epsilon=float(epsilon), alpha=float(alpha), max_iter=700)),
    ])
    final_model.fit(train_features, residual_target)
    residual_test = final_model.predict(test_features)

    return residual_oof, residual_test

def fold_mapes(pred):
    pred = np.clip(np.asarray(pred, dtype=float), 1e-9, None)
    return np.array([
        mean_absolute_percentage_error(y_values[valid_idx], pred[valid_idx])
        for _, valid_idx in final_splits
    ], dtype=float)

def masked_mape(pred, mask):
    pred = np.clip(np.asarray(pred, dtype=float), 1e-9, None)
    return float(mean_absolute_percentage_error(y_values[mask], pred[mask]))

def make_submission_frame(pred):
    pred = np.clip(np.asarray(pred, dtype=float), 1e-9, None)

    if ID_COL in test_raw.columns:
        return pd.DataFrame({ID_COL: test_raw[ID_COL].values, TARGET_COL: pred})

    return pd.DataFrame({ID_COL: np.arange(len(pred)), TARGET_COL: pred})

calibration_base_oof, calibration_base_test = build_base_predictions(0.705, "C0575")

calibration_residual_oof, calibration_residual_test = fit_predict_huber_residual(
    calibration_base_oof,
    calibration_base_test,
    final_hgb_oof[required_hgb_component],
    final_hgb_test[required_hgb_component],
    final_mlp_oof[required_mlp_component],
    final_mlp_test[required_mlp_component],
    epsilon=1.20,
    alpha=0.0005,
)

calibration_prediction_oof, _ = apply_huber_correction(
    calibration_base_oof,
    calibration_residual_oof,
    0.275,
    0.980,
    1.020,
)
calibration_prediction_test, _ = apply_huber_correction(
    calibration_base_test,
    calibration_residual_test,
    0.275,
    0.980,
    1.020,
)

base_oof, base_test = build_base_predictions(0.6225, "C060")

base_residual_oof, base_residual_test = fit_predict_huber_residual(
    base_oof,
    base_test,
    final_hgb_oof[required_hgb_component],
    final_hgb_test[required_hgb_component],
    final_mlp_oof[required_mlp_component],
    final_mlp_test[required_mlp_component],
    epsilon=1.20,
    alpha=0.0005,
)

base_raw_oof, _ = apply_huber_correction(base_oof, base_residual_oof, 0.345, 0.977, 1.023)
base_raw_test, _ = apply_huber_correction(base_test, base_residual_test, 0.345, 0.977, 1.023)

base_prediction_oof, _ = apply_ratio_cap(base_raw_oof, calibration_prediction_oof, 0.01375)
base_prediction_test, _ = apply_ratio_cap(base_raw_test, calibration_prediction_test, 0.01375)

check_prediction_array("base_prediction_oof", base_prediction_oof, len(y_values))
check_prediction_array("base_prediction_test", base_prediction_test, len(test_raw))

# The final selected setting applies a small Huber residual adjustment to the selected ensemble.
selected_residual_oof, selected_residual_test = fit_predict_huber_residual(
    base_prediction_oof,
    base_prediction_test,
    final_hgb_oof[required_hgb_component],
    final_hgb_test[required_hgb_component],
    final_mlp_oof[required_mlp_component],
    final_mlp_test[required_mlp_component],
    epsilon=1.20,
    alpha=0.0001,
)

selected_raw_oof, _ = apply_huber_correction(
    base_prediction_oof,
    selected_residual_oof,
    0.45,
    0.995,
    1.005,
)
selected_raw_test, _ = apply_huber_correction(
    base_prediction_test,
    selected_residual_test,
    0.45,
    0.995,
    1.005,
)

selected_oof, _ = apply_ratio_cap(selected_raw_oof, base_prediction_oof, 0.005)
selected_test, _ = apply_ratio_cap(selected_raw_test, base_prediction_test, 0.005)

selected_oof = check_prediction_array("selected_oof", selected_oof, len(y_values))
selected_test = check_prediction_array("selected_test", selected_test, len(test_raw))

output_prediction_path = OUTPUT_DIR / "prediction.csv"

final_prediction_frame = validate_prediction_frame(
    make_submission_frame(selected_test),
    expected_rows=len(test_raw),
    id_column=ID_COL,
    target_column=TARGET_COL,
)
final_prediction_frame.to_csv(output_prediction_path, index=False)

selected_scores = regression_scores(y_values, selected_oof)
fold_values = fold_mapes(selected_oof)

low_pressure_mask = y_values <= np.quantile(y_values, 0.10)
top_1_percent_mask = y_values >= np.quantile(y_values, 0.99)
top_2_percent_mask = y_values >= np.quantile(y_values, 0.98)

final_model_evidence = pd.DataFrame([{
    "final_model_name": "HistGradientBoosting and MLP ensemble with cross-validated Huber residual calibration",
    "selected_model_setting": "epsilon 1.20, alpha 0.0001, shrinkage 0.45, cap 0.005",
    "selected_oof_mape": float(selected_scores["mape"]),
    "selected_oof_r2": float(selected_scores["r2"]),
    "selected_oof_rmse": float(selected_scores["rmse"]),
    "selected_oof_mae": float(selected_scores["mae"]),
    "fold_mape_mean": float(fold_values.mean()),
    "fold_mape_std": float(fold_values.std()),
    "low_pressure_mape": float(masked_mape(selected_oof, low_pressure_mask)),
    "top_1_percent_mape": float(masked_mape(selected_oof, top_1_percent_mask)),
    "top_2_percent_mape": float(masked_mape(selected_oof, top_2_percent_mask)),
    "prediction_csv_path": str(output_prediction_path.relative_to(PROJECT_ROOT)),
    "final_prediction_csv_created": bool(output_prediction_path.exists()),
}])

write_final_table(final_model_evidence, "04_final_model_evidence.csv")

final_submission = pd.read_csv(output_prediction_path)
final_values = pd.to_numeric(final_submission[TARGET_COL], errors="coerce")

prediction_integrity = pd.DataFrame([{
    "file": str(output_prediction_path.relative_to(PROJECT_ROOT)),
    "rows": int(len(final_submission)),
    "expected_rows": int(len(test_raw)),
    "columns": ", ".join(list(final_submission.columns)),
    "has_nan": bool(final_submission.isna().any().any()),
    "non_positive_predictions": int((final_values <= 0).sum()),
    "numeric_prediction_min": float(final_values.min()),
    "numeric_prediction_mean": float(final_values.mean()),
    "numeric_prediction_median": float(final_values.median()),
    "numeric_prediction_p95": float(final_values.quantile(0.95)),
    "numeric_prediction_p99": float(final_values.quantile(0.99)),
    "numeric_prediction_p995": float(final_values.quantile(0.995)),
    "numeric_prediction_max": float(final_values.max()),
    "row_count_ok": bool(len(final_submission) == len(test_raw)),
}])

write_final_table(prediction_integrity, "05_prediction_integrity_check.csv")

plt.figure(figsize=(8, 4))
plt.hist(final_values, bins=60)
plt.xlabel("Predicted Target Pressure (bar)")
plt.ylabel("Count")
plt.title("Final Prediction Distribution")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "final_prediction_distribution.png", dpi=150)
plt.show()

print("Final model export complete.")
print("Selected final model:", final_model_evidence.loc[0, "final_model_name"])
print("Prediction export:", output_prediction_path.relative_to(PROJECT_ROOT))

## 10. Final workflow summary

This section records the main modelling steps completed in the notebook and saves a short summary table for the project documentation.

In [ ]:
workflow_summary = pd.DataFrame([
    {
        "section": "Data preparation",
        "summary": "The training and test data are loaded, checked, cleaned, and prepared for modelling."
    },
    {
        "section": "Feature preparation",
        "summary": "The notebook creates geometry, ratio, pressure, temperature, obstacle, and sensor-position features that describe the BLEVE setting."
    },
    {
        "section": "Model comparison",
        "summary": "Several model families are evaluated using cross-validation before selecting the final modelling approach."
    },
    {
        "section": "Model tuning",
        "summary": "The selected model family is tuned using cross-validation, including nearby HGB settings, MLP support settings, blend weights, and residual calibration settings."
    },
    {
        "section": "Evaluation",
        "summary": "Model performance is reported with MAPE, R2, RMSE, and MAE, together with fold-level validation evidence."
    },
    {
        "section": "Final prediction",
        "summary": "The final selected model is used to generate the prediction file, and the exported file is checked for row count, missing values, and valid positive predictions."
    },
])

workflow_summary.to_csv(TABLE_DIR / "06_workflow_summary.csv", index=False)
display(workflow_summary)